# Cheat sheet: list-like columns, cleaning and counting

The examples use made-up data (a `colors` column), so you can adapt them to your exercise.

## 1. Look at the data first

| Goal | Code |
|---|---|
| See the first cell | `df["colors"].iloc[0]` |
| See its type | `type(df["colors"].iloc[0])` |
| Column type | `df["colors"].dtype` |
| See a few cells | `df["colors"].head()` |

Check this **before** any cleaning. A cell can be a real list (`["red", "blue"]`) or a string that looks like a list (`"['red', 'blue']"`). They need different tools.

## 2. Turn a string into a real list

| Case | Code |
|---|---|
| The cell is already a list | nothing to do |
| String that looks like a Python list | `import ast` then `df["colors"].apply(ast.literal_eval)` |
| Simple text with a separator | `df["colors"].str.split(",")` |
| Remove characters at the two ends | `df["colors"].str.strip("[]")` |
| Replace characters inside | `df["colors"].str.replace("'", "", regex=False)` |
| Chain the steps | `df["colors"].str.strip("[]").str.replace("'", "", regex=False).str.split(",")` |

Notes:

- `ast.literal_eval` safely reads a string that looks like a Python list. It is the cleanest tool when the text is a proper list. Test it on one cell first.
- `strip("abc")` removes **any** of the characters `a`, `b`, `c` at the **two ends**. It does not remove a word, and it does not touch the middle.
- `.str.split(",")` gives a list per cell. The items often keep spaces around them.

## 3. Clean each item of a list

| Goal | Code |
|---|---|
| Remove spaces around each item | `[item.strip() for item in my_list]` |
| Same, on a whole column | `df["colors"].apply(lambda items: [item.strip() for item in items])` |
| Remove quotes and spaces | `item.strip(" '\"")` |
| Lowercase | `item.lower()` |

Why it matters: `"red"` and `" red"` are two different values for Python, so they would be counted separately.

## 4. Count in one list (per row)

| Goal | Code |
|---|---|
| Dictionary of counts, by hand | loop, then `if item in counts: counts[item] += 1`, else `counts[item] = 1` |
| With the standard library | `from collections import Counter` then `Counter(my_list)` |
| Apply to each row | `df["colors"].apply(my_function)` |

`apply` calls the function **once per row**. You get one result per row.

## 5. Count across the whole column

| Goal | Code |
|---|---|
| One row per item (explode) | `df["colors"].explode()` |
| Count each item | `df["colors"].explode().value_counts()` |
| Top 5 | `df["colors"].explode().value_counts().head(5)` |
| Number of different items | `df["colors"].explode().nunique()` |
| With a loop and one dictionary | a `for` over the column, and a `for` over each list, updating the same dictionary |
| With `Counter` | create `Counter()` once, then `total.update(my_list)` for each list |
| Dictionary sorted by count | `sorted(counts.items(), key=lambda pair: pair[1], reverse=True)` |

`explode` puts each item of each list on its own row. After that, normal column tools like `value_counts` work.

## 6. Other useful tools

| Goal | Code |
|---|---|
| Number of items per row | `df["colors"].apply(len)` or `df["colors"].str.len()` |
| Rows containing an item | `df[df["colors"].apply(lambda items: "red" in items)]` |
| Remove empty lists | `df[df["colors"].str.len() > 0]` |

## 7. Per row or whole column?

| You want | Use |
|---|---|
| One result for each row | `apply` with a function |
| One result for the whole column | `explode()` then `value_counts()`, or a loop with one shared dictionary |

## 8. Common mistakes

| Error | Cause | Fix |
|---|---|---|
| `NameError: name '...' is not defined` | The name in `apply(...)` is not the name in `def` | Use the same name |
| `list("abc")` gives `["a", "b", "c"]` | `list()` on a string splits it into characters | Check the type first |
| `NaN` after `.str...` | The column already holds lists, or the cell was run twice | Reload the data, run the cleaning once |
| Same item counted twice | Spaces or quotes around values | Clean each item with `strip()` |
| `strip("'u")` cuts letters in names | `strip` removes characters, not words | Use `replace` or `literal_eval` |
| `apply` result is not saved | No assignment | `df["new"] = df["colors"].apply(...)` |
| `.str` does not work | The cells are lists, not text | `.str.split` and `.str.strip` are for text. For lists use `apply`, or `explode` |

## 9. Order of work

1. Look at one cell and its type.
2. Turn it into a real list.
3. Clean each item (spaces, quotes).
4. Test your function on **one** list.
5. Apply it with `apply`, or use `explode` and `value_counts` for the whole column.
6. Check the result on a few rows.

## 10. PEP 8 and PEP 257

- One-line docstring in triple double quotes, with a period.
- Names in `snake_case`, with a clear meaning (`actors`, not `a_list`).
- No spaces on empty lines, and no spaces at the end of lines.
- Two blank lines around functions.
- Imports at the top of the file or notebook, one per line.

# Mission

Still using this dataset: http://bit.ly/imdbratings
- Which actor is mentioned the most often?


In [35]:
import pandas as pd

link = 'https://raw.githubusercontent.com/justmarkham/pandas-videos/master/data/imdb_1000.csv'
df = pd.read_csv(link)
df

,star_rating,title,content_rating,genre,duration,actors_list
0,9.3,The Shawshank Redemption,R,Crime,142,"[u'Tim Robbins', u'Morgan Freeman', u'Bob Gunt..."
1,9.2,The Godfather,R,Crime,175,"[u'Marlon Brando', u'Al Pacino', u'James Caan']"
2,9.1,The Godfather: Part II,R,Crime,200,"[u'Al Pacino', u'Robert De Niro', u'Robert Duv..."
3,9.0,The Dark Knight,PG-13,Action,152,"[u'Christian Bale', u'Heath Ledger', u'Aaron E..."
4,8.9,Pulp Fiction,R,Crime,154,"[u'John Travolta', u'Uma Thurman', u'Samuel L...."
...,...,...,...,...,...,...
974,7.4,Tootsie,PG,Comedy,116,"[u'Dustin Hoffman', u'Jessica Lange', u'Teri G..."
975,7.4,Back to the Future Part III,PG,Adventure,118,"[u'Michael J. Fox', u'Christopher Lloyd', u'Ma..."
976,7.4,Master and Commander: The Far Side of the World,PG-13,Action,138,"[u'Russell Crowe', u'Paul Bettany', u'Billy Bo..."
977,7.4,Poltergeist,PG,Horror,114,"[u'JoBeth Williams', u""Heather O'Rourke"", u'Cr..."


In [36]:
type(df["actors_list"].iloc[0])

str

In [37]:
def get_actor_dict(a_list):
    actor_dict = {}
    actors_list = list(a_list)
    
    for actor in actors_list:
        if actor in actor_dict:
            actor_dict[actor] += 1
        else:
            actor_dict[actor] = 1
    return actor_dict

In [38]:
df["actors_list"].iloc[0]
type(df["actors_list"].iloc[0])

str

In [50]:
import ast

In [52]:
df["actors_list"].apply(ast.literal_eval).explode().value_counts().head(3)

actors_list
Robert De Niro    18
Clint Eastwood    14
Brad Pitt         14
Name: count, dtype: int64

In [54]:
# REGEX MAIS CHIANT
actor_counts = (df["actors_list"].str.findall(r"(?:\[|, )u['\"](.*?)['\"](?=,|\])").explode().value_counts())
actor_counts.head(3)

actors_list
Robert De Niro    18
Clint Eastwood    14
Brad Pitt         14
Name: count, dtype: int64